# 样本报告分析日志：章丘女郎山

本 notebook 记录对《章丘女郎山》发掘报告 Markdown 文本的结构化观察，用于验证和改进 `tomb_extraction_pipeline.ipynb` 的抽取字段与解析策略。

## 1. 报告概况

- **报告名称**：章丘女郎山
- **出版信息**：济南市考古研究所 编著，科学出版社，2012 年（2013 年 1 月第 1 版）
- **发掘时间**：2009 年
- **墓葬年代**：汉、唐、宋、金、元、明、清（共 400 余座墓葬）
- **地理位置**：山东省济南市章丘区女郎山墓地
- **主要特征**：
  - 按时代分章，每章再按墓葬形制分类
  - 每座墓葬有独立条目，包含位置、开口层位、形制、方向、墓圹尺寸、填土、二层台、壁龛、葬具、人骨、葬式、头向、随葬品及位置、插图编号
  - 常见形制：土坑竖穴墓、土坑竖穴砖椁墓、土坑洞室墓、砖室墓、舟形墓、砖石室墓
  - 常见随葬品：陶罐、陶壶、铜钱、铜镜、瓷瓶、瓷碗等
  - 人骨保存普遍较差，多仅余灰痕或粉化
- **年代判断依据**：报告本身已按时代分章；具体墓葬依赖出土钱币（五铢钱、开元通宝、崇宁通宝）、器物类型学及墓葬结构断代
- **文献来源**：科学出版社《章丘女郎山》考古发掘报告

## 2. 逐墓条目示例（M3）

> M3位于第Ⅰ发掘区东北部，T1929东北部，开口于耕土层下，距地表0.2米。墓葬形制为土坑竖穴墓，方向94°。墓圹平面呈长方形，直壁、平底，长2.7、宽2、深1.3米。填土为黄褐色花土，结构疏松，含料姜石。墓底铺一层青砖：中间1列纵向平铺，两边横向对缝平铺。所用青砖单面有绳纹，长40、宽16、厚8厘米。墓底一周有熟土二层台，宽0.3~0.5、高0.44米。墓底中部有葬具1棺，已朽，平面呈长方形，长2.1、宽1米。棺内有人骨架1具，保存较差，葬式为仰身直肢，头向东。随葬品有灰陶罐1件，放于脚端二层台上（图2-1）。

可见一条完整的墓葬条目已包含：位置、开口层位、形制、方向、墓圹平面/尺寸、填土、铺地、二层台、葬具、人骨、葬式、头向、随葬品、图号。

## 3. 字段映射检查

| 管线字段 | 报告中是否直接可抽取 | 说明 |
|---|---|---|
| tomb_id | ✅ | M3、M23 等编号 |
| site_name | ✅ | 章丘女郎山 |
| location | ✅ | 第Ⅰ发掘区东北部 T1929 东北部 |
| tomb_type | ✅ | 土坑竖穴墓、砖室墓、土坑洞室墓等 |
| orientation | ✅ | 方向 94° 等 |
| passage | ✅ | 有墓道时描述 |
| chamber_shape | ✅ | 长方形、梯形、刀把形等 |
| chamber_size | ✅ | 长 2.7、宽 2、深 1.3 米 |
| sealing | ⚠️ | 砖室墓有“青砖砌筑封门墙” |
| coffin | ✅ | 1 棺，已朽 |
| platform | ✅ | 生土/熟土二层台 |
| niche | ✅ | 壁龛位置、尺寸 |
| flooring | ✅ | 青砖铺地 |
| burial_position | ✅ | 仰身直肢 |
| head_direction | ✅ | 头向东/向北 |
| bone_preservation | ✅ | 保存较差、粉化严重 |
| artifact_distribution | ✅ | 脚端二层台上、墓底南端等 |
| artifact_types | ✅ | 陶罐、陶壶、铜钱、铜镜 |
| artifact_count | ✅ | 1 件、2 件、4 件（组） |
| reported_date | ✅ | 报告按时代分章给出 |
| reported_ethnicity | ❌ | 本报告不涉及族属判断 |
| note | ⚠️ | 图文冲突、缺失信息、打破关系 |

结论：这份报告覆盖了 prompt 中绝大多数字段，是理想的训练/测试样本；族属字段在此不适用，但在魏晋十六国项目中是关键字段。

In [ ]:
from pathlib import Path

# 样本报告已放入 markdown/ 目录
report_path = Path("markdown/章丘女郎山.md")
assert report_path.exists(), f"{report_path} 不存在，请确认样本已放入 markdown/ 目录"

text = report_path.read_text(encoding="utf-8")
print(f"报告总字符数：{len(text):,}")
print(f"报告总行数：{text.count(chr(10)):,}")

In [ ]:
import re
import pandas as pd

# 从目录中提取各时代章节下的墓葬编号
dynasty_sections = {
    "汉代": r"第二章 汉代墓葬.*?(?=第三章 唐代墓葬)",
    "唐代": r"第三章 唐代墓葬.*?(?=第四章 宋代墓葬)",
    "宋代": r"第四章 宋代墓葬.*?(?=第五章 金代墓葬)",
    "金代": r"第五章 金代墓葬.*?(?=第六章 元代墓葬)",
    "元代": r"第六章 元代墓葬.*?(?=第七章 明代墓葬)",
    "明代": r"第七章 明代墓葬.*?(?=第八章 清代墓葬)",
    "清代": r"第八章 清代墓葬.*?(?=第九章 对女郎山各时期墓葬的初步认识)",
}

counts = {}
for dynasty, pattern in dynasty_sections.items():
    m = re.search(pattern, text, re.DOTALL)
    if m:
        # 目录中形如：一、M3……（13）
        tomb_ids = re.findall(r"[一二三四五六七八九十百]+、M(\d+)", m.group(0))
        counts[dynasty] = len(tomb_ids)
    else:
        counts[dynasty] = 0

df = pd.DataFrame.from_dict(counts, orient="index", columns=["墓葬数量"])
df.index.name = "时代"
df = df.sort_values("墓葬数量", ascending=False)
df

In [ ]:
# 提取所有以 Mxxx 开头的墓葬描述段落（单行）
tomb_entries = []
for line in text.splitlines():
    if re.match(r"M\d+(位于|.*形制)", line):
        tomb_entries.append(line)

print(f"共提取到 {len(tomb_entries)} 个以 Mxxx 开头的墓葬描述段落")
print("\n前 3 条示例：")
for e in tomb_entries[:3]:
    print(e[:250], "...")

## 4. 对管线的启示

1. **输入格式稳定**：这种科学出版社正式报告的结构极其规范，适合 LLM 批量抽取。
2. **年代字段已存在**：报告作者已做时代分期，可直接作为 `reported_date`；但管线仍需保留 `dated_material` 和 `dating_basis` 以记录断代证据。
3. **族属字段缺失**：本报告不涉及族属，而你的核心项目是魏晋十六国，因此 prompt 中 `reported_ethnicity` / 族属判断必须突出。
4. **图文对应较好**：本报告中插图目录与文字条目基本对应，但其他简报可能出现“图与文不对应”，`note` 字段应强制 LLM 标注。
5. **数量表达多样**：“1 件”“2 件”“4 件（组）”“铜钱 5 枚（1 组）”等，需要在 `artifact_count` / `artifact_combination` 中做标准化。
6. **打破关系值得记录**：如“被 M338、M339 打破”，可纳入 `note` 或新增 `stratigraphic_relationship` 字段。

## 5. 下一步

- [ ] 用本报告作为第一批测试数据，跑通 `tomb_extraction_pipeline.ipynb`
- [ ] 检查 LLM 抽取结果中 `chamber_size` 和 `artifact_count` 的格式是否统一
- [ ] 针对魏晋十六国材料，在 prompt 中增加族属判断、文化因素、断代依据的权重
- [ ] 建立“图文冲突”标注规则，训练 LLM 识别图注与正文不一致的情况
- [ ] 将本报告的地理坐标（章丘女郎山）加入 `location` 字段，方便后续 QGIS 落点
- [ ] 考虑新增 `stratigraphic_relationship`（打破关系）字段